In [ ]:
import os
import chex
import jax.numpy as jnp
import natpy as nat

from cnb.config import ROOT
from cnb.physics import Physics
from cnb.sim_data import SimData
from cnb.units import Params

# Data paths in this notebook are relative to the repo root.
os.chdir(ROOT)

%load_ext autoreload
%autoreload 2


import pandas as pd

sim_name = f"Dopri5_1k"
sim_folder = f"runs/{sim_name}"
fig_folder = f"figures/{sim_name}"
Cl_folder = f"data/primordial/Cls"
nu_m_range = jnp.load(f"{sim_folder}/neutrino_massrange_eV.npy")
nu_m_picks = jnp.array([0.01, 0.05, 0.1, 0.2, 0.3])*Params.eV
simdata = SimData(sim_folder)

z_int_steps = jnp.load(f"{sim_folder}/z_int_steps.npy")
s_int_steps = jnp.load(f"{sim_folder}/s_int_steps.npy")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# PMNS matrix elements (average of 3sigma ranges)
pmns_elements = np.array([
    [(0.801 + 0.845)/2, (0.513 + 0.579)/2, (0.143 + 0.156)/2],
    [(0.233 + 0.507)/2, (0.461 + 0.694)/2, (0.631 + 0.778)/2],
    [(0.261 + 0.526)/2, (0.471 + 0.701)/2, (0.611 + 0.761)/2]
])

# Square the elements (oscillation probabilities)
pmns_squared = pmns_elements**2
print(pmns_squared.shape)
print(np.sum(pmns_squared, axis=-1))

# Mass differences (artificially adjusted for illustration)
delta_m21_sq = 0.8  # Artificially increased solar mass difference
delta_m31_sq_normal = 2.5   # Atmospheric mass difference (normal ordering)
delta_m32_sq_inverted = -2.5  # Inverted ordering

# Calculate mass squared values (relative to lightest neutrino)
# Normal ordering: m1 < m2 < m3
m1_sq_normal = 0
m2_sq_normal = delta_m21_sq
m3_sq_normal = delta_m21_sq + delta_m31_sq_normal

# Inverted ordering: m3 < m1 < m2
m3_sq_inverted = 0
m1_sq_inverted = abs(delta_m32_sq_inverted)
m2_sq_inverted = abs(delta_m32_sq_inverted) + delta_m21_sq

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Colors for each flavor
colors = ['#FF6B6B', "#A545D1", "#4E6CCD", ]
flavor_labels = [r'$\nu_e$', r'$\nu_\mu$', r'$\nu_\tau$']
m_text_size = 26

# Bar properties
bar_width = 2
bar_height = 0.3

# Positions
normal_x = -0.9
inverted_x = 1.7

# Normal ordering
normal_masses = [m1_sq_normal, m2_sq_normal, m3_sq_normal]
for i, mass_sq in enumerate(normal_masses):
    y_pos = mass_sq
    
    # Create stacked bar segments for each flavor contribution
    left_edge = normal_x - bar_width/2
    cumulative_width = 0
    
    for j in range(3):  # For each flavor
        segment_width = bar_width * pmns_squared[j, i]
        rect = Rectangle((left_edge + cumulative_width, y_pos - bar_height/2), 
                        segment_width, bar_height, 
                        facecolor=colors[j], alpha=0.8, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        cumulative_width += segment_width
    
    # Add mass eigenstate label
    ax.text(normal_x - bar_width/2 - 0.3, y_pos, rf'$\nu_{i+1}$', 
            ha='right', va='center', fontsize=m_text_size, fontweight='bold')

# Inverted ordering
inverted_masses = [m1_sq_inverted, m2_sq_inverted, m3_sq_inverted]
inverted_indices = [0, 1, 2]  # Mass eigenstate indices
for idx, (i, mass_sq) in enumerate(zip(inverted_indices, inverted_masses)):
    y_pos = mass_sq
    
    # Create stacked bar segments for each flavor contribution
    left_edge = inverted_x - bar_width/2
    cumulative_width = 0
    
    for j in range(3):  # For each flavor
        segment_width = bar_width * pmns_squared[j, i]
        rect = Rectangle((left_edge + cumulative_width, y_pos - bar_height/2), 
                        segment_width, bar_height, 
                        facecolor=colors[j], alpha=0.8, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        cumulative_width += segment_width
    
    # Add mass eigenstate label
    ax.text(inverted_x + bar_width/2 + 0.3, y_pos, rf'$\nu_{i+1}$', 
            ha='left', va='center', fontsize=m_text_size, fontweight='bold')


# Add mass difference annotations between bars
# Normal ordering
mid_12_normal = (m1_sq_normal + m2_sq_normal) / 2
mid_23_normal = (m2_sq_normal + m3_sq_normal) / 2

ax.text(normal_x, mid_12_normal, r'$\Delta m^2_{21}$', 
        ha='center', va='center', fontsize=m_text_size, fontweight='bold')
ax.text(normal_x, mid_23_normal, r'$\Delta m^2_{32}$', 
        ha='center', va='center', fontsize=m_text_size, fontweight='bold')

# Inverted ordering
mid_31_inverted = (m3_sq_inverted + m1_sq_inverted) / 2
mid_12_inverted = (m1_sq_inverted + m2_sq_inverted) / 2

ax.text(inverted_x, mid_31_inverted, r'$\Delta m^2_{31}$', 
        ha='center', va='center', fontsize=m_text_size, fontweight='bold')
ax.text(inverted_x, mid_12_inverted, r'$\Delta m^2_{21}$', 
        ha='center', va='center', fontsize=m_text_size, fontweight='bold')

# Set axis limits
ax.set_xlim(-4, 4)
ax.set_ylim(-0.5, 4)

# Remove all axes, ticks, and spines
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# Add ordering labels beneath each diagram
ax.text(normal_x, 3.8, r'$\textbf{Normal Ordering}$', ha='center', va='center', 
        fontsize=24, fontweight='bold')
ax.text(inverted_x, 3.8, r'$\textbf{Inverted Ordering}$', ha='center', va='center', 
        fontsize=24, fontweight='bold')

# Add legend
legend_elements = [
    mpatches.Patch(color=colors[i], label=flavor_labels[i], alpha=0.8) 
    for i in range(3)
]
ax.legend(
    handles=legend_elements, loc='center left', fontsize=30, 
    frameon=False, fancybox=True)

plt.tight_layout()

plt.savefig(f"neutrino_mass_ordering.pdf", bbox_inches="tight")
plt.show()

# Print some key information
print("PMNS Matrix Elements² (oscillation probabilities):")
for i, flavor in enumerate(['e', 'μ', 'τ']):
    print(f"ν{flavor}: {pmns_squared[i]}")
print(f"\nMass differences (artificially scaled for illustration):")
print(f"Δm²₂₁ = {delta_m21_sq} (scaled)")
print(f"Δm²₃₁ (normal) = {delta_m31_sq_normal} (scaled)")
print(f"Δm²₃₂ (inverted) = {delta_m32_sq_inverted} (scaled)")

In [ ]:
def q_to_velocity_bounds(q_min, q_max, m_eV, args):
    p_min = q_min * args.T_CNB
    p_max = q_max * args.T_CNB

    v_min = (p_min / m_eV) / (args.km/args.s)
    v_max = (p_max / m_eV) / (args.km/args.s)

    print(v_min, v_max)
    # return v_min, v_max


for m in (0.01, 0.3):
    q_to_velocity_bounds(
        q_min=0.399, 
        q_max=9.374, 
        m_eV=m, 
        args=Params())

In [ ]:
(46*nat.Glyr).convert(nat.Gpc)

### CNB in solar region.

In [ ]:
ES_dist_av = 150.27e6*Params.km
nu_vel_max = 0.05*Params.c
nu_ES_time = ES_dist_av/nu_vel_max
time_unit = Params.s*60*60*24
print(nu_ES_time/time_unit)

### Neutrino mass hierarchies.

In [ ]:
### ====================================== ###
### Global loop over neutrino mass choices ###
### ====================================== ###

# We later always select the first of the 3 masses, to generate the Cl's.
# In that way we get Cl's precisely for first mass. Class still needs 3 masses.

nu_mass_set0 = Physics.neutrino_masses(
    m_lightest=0.00001, ordering='NO', args=Params())

# To get 0.01 eV
nu_mass_set1 = Physics.neutrino_masses(
    m_lightest=0.01, ordering='NO', args=Params())

# To get 0.05 eV
nu_mass_set2 = Physics.neutrino_masses(
    m_lightest=0.05, ordering='NO', args=Params())

# To get 0.1 eV
nu_mass_set3 = Physics.neutrino_masses(
    m_lightest=0.1, ordering='NO', args=Params())

# To get 0.2 eV
nu_mass_set4 = Physics.neutrino_masses(
    m_lightest=0.2, ordering='NO', args=Params())

# To get 0.3 eV
nu_mass_set5 = Physics.neutrino_masses(
    m_lightest=0.3, ordering='NO', args=Params())

nu_masses_sets = [
    nu_mass_set0.tolist(),
    nu_mass_set1.tolist(),
    nu_mass_set2.tolist(),
    nu_mass_set3.tolist(),
    nu_mass_set4.tolist(),
    nu_mass_set5.tolist(),
]

nu_masses = nu_masses_sets[0]
print(nu_masses)

# Always selecting first mass out of the 3
m_idx = 0
m_val = nu_masses[m_idx]

print(fr"*** Generating Cl for $m_\nu$={m_val} eV ***")

In [ ]:
@chex.dataclass
class Test:
    # Temperature of CMB
    T_CMB: float = 2.72548

    # N_eff parameter, using results of https://doi.org/10.1103/PhysRevD.108.L121301
    N_eff: float = 3.043

    # Temperature of CNB
    T_CNB = jnp.power(4/11, 1/3)*T_CMB * jnp.power(N_eff/3, 1/4)

    # Temperature of CNB
    def T_CNB_fct(N_eff=N_eff, T_CMB=T_CMB):
        return jnp.power(4/11, 1/3)*T_CMB * jnp.power(N_eff/3, 1/4)


print(Test.T_CNB_fct())
print(Test.T_CNB_fct(N_eff=3.0))

print("END")